# Warsztat 4: Rozmowa z kupcem sterowanym przez LLM (Google Gemma)

### Cel ćwiczenia

W grach fabularnych (RPG) gracze chętnie prowadzą swobodne rozmowy z postaciami niezależnymi (NPC), jednak stan świata gry — liczba monet w sakiewce, zawartość ekwipunku czy zapasy w sklepie — musi podlegać ścisłym, deterministycznym regułom.

Waszym zadaniem jest zbudowanie tekstowego systemu dialogu z kupcem w grze przy użyciu lokalnego modelu instrukcyjnego **Google Gemma (`gemma2:2b`)** uruchamianego przez **Ollama**:
1. Gracz wpisuje dowolną wypowiedź (np. pytanie o towar, prośbę o zakup mikstury, niejasną uwagę lub żądanie nieobsługiwanej akcji).
2. Model językowy odgrywa rolę kupca, rozpoznaje intencję gracza i zwraca ustrukturyzowaną odpowiedź w formacie **JSON** zawierającą pola `"reply"` (wypowiedź kupca) oraz `"action"` (proponowana akcja: `"talk"` lub `"buy_potion"`).
3. **Program w języku Python** waliduje strukturę JSON, sprawdza czy akcja należy do listy dozwolonych i **samodzielnie rozstrzyga**, czy transakcja jest możliwa (weryfikuje liczbę monet gracza oraz stan magazynu kupca), a następnie aktualizuje stan gry albo odrzuca zakup.

Celem warsztatu jest poznanie bezpiecznego wzorca integracji swobodnej generacji LLM ze ściśle kontrolowaną maszyną stanów gry.

### Stan początkowy i reguły gry

- **Stan początkowy (`GameState`):**
  - Gracz posiada **15 monet** (`coins = 15`).
  - Kupiec posiada w magazynie **2 mikstury** (`potions = 2`).
  - Każda mikstura kosztuje **10 monet** (`potion_price = 10`).
- **Dozwolone akcje (`ALLOWED_ACTIONS`):**
  - `"talk"` — zwykła rozmowa, pytanie o ofertę, brak transakcji lub odrzucenie prośby przez kupca.
  - `"buy_potion"` — intencja zakupu dokładnie **jednej** mikstury.
- **Format wyjściowy modelu (JSON):**
  ```json
  {
    "reply": "Tekst odpowiedzi kupca w świecie gry...",
    "action": "talk"
  }
  ```
- **Złota zasada architektury:** Saldo monet, cena i zapas mikstur są utrzymywane **wyłącznie przez program**. Niepoprawny JSON, nieznana akcja spoza listy lub przekroczenie czasu oczekiwania (`timeout`) **nigdy nie zmieniają stanu gry**. Komunikat potwierdzający lub odrzucający transakcję powstaje na podstawie decyzji programu, a nie samej wypowiedzi modelu.

### 1. Przygotowanie środowiska i podział odpowiedzialności (0–10 min)

W tej części:
1. Omawiamy w zespole podział odpowiedzialności między model językowy a logikę gry.
2. Otwieramy plik `sprawozdanie_szablon.md`, wpisujemy skład zespołu, nazwę modelu (`gemma2:2b`) oraz wnioski z dyskusji w sekcji 2.
3. Importujemy funkcje pomocnicze z modułu `merchant_utils.py` i sprawdzamy połączenie z lokalnym modelem Google Gemma.

In [1]:
import json
from typing import Literal
from pydantic import BaseModel, Field

from merchant_utils import (
    ALLOWED_ACTIONS,
    DEFAULT_MODEL,
    GameState,
    TurnResult,
    call_llm,
    load_scenarios,
    print_markdown_table,
    print_results_table,
    show_dialogue_screen,
)

MODEL_NAME = DEFAULT_MODEL  # Domyślnie: "gemma2:2b"
TEMPERATURE = 0.2
SEED = 42
TIMEOUT_SEC = 20.0

initial_state = GameState(coins=15, potions=2, potion_price=10)
print(f"Wybrany model: {MODEL_NAME}")
print(f"Dozwolone akcje: {ALLOWED_ACTIONS}")
print(f"Stan początkowy gry: {initial_state.format_short()}")

Wybrany model: gemma2:2b
Dozwolone akcje: ('talk', 'buy_potion')
Stan początkowy gry: monety=15, mikstury=2 (cena=10)


In [2]:
# Szybki test połączenia z modelem Google Gemma przez Ollama
raw_ping, ping_time, is_timeout, err = call_llm(
    system_prompt='Zwróć JSON: {"status": "ok"}',
    user_message="Test połączenia",
    model=MODEL_NAME,
    temperature=TEMPERATURE,
    seed=SEED,
    timeout_sec=TIMEOUT_SEC,
)
print(f"Odpowiedź testowa ({ping_time:.2f} s, timeout={is_timeout}, błąd={err}): {raw_ping}")

Odpowiedź testowa (9.34 s, timeout=False, błąd=None): {"status": "ok"}


### 2. Instrukcja dla kupca i schemat odpowiedzi JSON (10–30 min)

Biblioteka `ollama` oraz model **Google Gemma (`gemma2:2b`)** obsługują wymuszanie odpowiedzi zgodnej ze schematem JSON (*Structured Outputs* przez parametr `format`). Definiujemy schemat za pomocą klasy `Pydantic`, a następnie budujemy instrukcję systemową (`system_prompt`), która przekazuje modelowi:
- kim jest (kupiec Barnaba sprzedający wyłącznie mikstury uzdrawiające),
- jaki jest aktualny stan gry (monety gracza, liczba dostępnych mikstur w magazynie, cena jednej mikstury),
- jakie są dozwolone akcje (`"talk"` oraz `"buy_potion"`) i kiedy należy użyć każdej z nich,
- wymóg odpowiadania po polsku w formacie JSON z polami `"reply"` i `"action"`.

In [3]:
class MerchantResponse(BaseModel):
    """Schemat odpowiedzi wymagany od modelu językowego."""

    reply: str = Field(description="Wypowiedź kupca skierowana do gracza po polsku.")
    action: Literal["talk", "buy_potion"] = Field(
        description="Rozpoznana intencja/akcja: 'talk' dla rozmowy lub odmowy, 'buy_potion' gdy gracz chce kupić miksturę."
    )


RESPONSE_SCHEMA = MerchantResponse.model_json_schema()
print("Wygenerowany schemat JSON dla Ollama:")
print(json.dumps(RESPONSE_SCHEMA, indent=2, ensure_ascii=False))

Wygenerowany schemat JSON dla Ollama:
{
  "description": "Schemat odpowiedzi wymagany od modelu językowego.",
  "properties": {
    "reply": {
      "description": "Wypowiedź kupca skierowana do gracza po polsku.",
      "title": "Reply",
      "type": "string"
    },
    "action": {
      "description": "Rozpoznana intencja/akcja: 'talk' dla rozmowy lub odmowy, 'buy_potion' gdy gracz chce kupić miksturę.",
      "enum": [
        "talk",
        "buy_potion"
      ],
      "title": "Action",
      "type": "string"
    }
  },
  "required": [
    "reply",
    "action"
  ],
  "title": "MerchantResponse",
  "type": "object"
}


In [4]:
def build_merchant_prompt(state: GameState) -> str:
    """Buduje instrukcję systemową dla kupca uwzględniającą aktualny stan gry."""
    prompt = f"""Jesteś kupcem Barnabą prowadzącym sklep z miksturami w grze RPG.
Odpowiadasz krótko (1-2 zdania), w klimacie fantasy, wyłącznie po polsku.

AKTUALNY STAN GRY:
- Monety gracza: {state.coins}
- Dostępne mikstury w Twoim magazynie: {state.potions} szt.
- Cena jednej mikstury: {state.potion_price} monet

ZASADY SKLEPU I WYBORU AKCJI (pole "action"):
1. Masz w ofercie wyłącznie mikstury uzdrawiające w cenie {state.potion_price} monet za sztukę. Jedna transakcja to zakup dokładnie 1 mikstury.
2. Nie skupujesz przedmiotów od gracza, nie udzielasz kredytu, nie dajesz zniżek i nie oferujesz innych towarów.
3. Ustaw pole "action" na "buy_potion" TYLKO WTEDY, gdy gracz wyraźnie prosi o zakup mikstury.
   Uwaga: jeśli gracz ma mniej niż {state.potion_price} monet lub w magazynie jest 0 mikstur, grzecznie poinformuj w "reply", że zakup jest niemożliwy (brak złota lub towaru).
4. Ustaw pole "action" na "talk", gdy gracz zadaje pytanie o ofertę/cenę, wita się, wypowiada się niejasno lub żąda nieobsługiwanej czynności (np. sprzedaży miecza, targowania się).

Zwróć wyłącznie poprawny obiekt JSON z polami "reply" (tekst odpowiedzi kupca) oraz "action" ("talk" lub "buy_potion")."""
    return prompt

In [5]:
# TODO: Sprawdź działanie modelu na zwykłym pytaniu o ofertę oraz na prośbie o zakup mikstury.
test_prompt = build_merchant_prompt(initial_state)

for sample_input in [
    "Dzień dobry! Co sprzedajesz i ile to kosztuje?",
    "Chcę kupić jedną miksturę uzdrawiającą.",
]:
    raw_out, elapsed, timed_out, error = call_llm(
        system_prompt=test_prompt,
        user_message=sample_input,
        model=MODEL_NAME,
        schema=RESPONSE_SCHEMA,
        temperature=TEMPERATURE,
        seed=SEED,
        timeout_sec=TIMEOUT_SEC,
    )
    print(f"Gracz: {sample_input}")
    print(f"Surowy JSON z modelu ({elapsed:.2f} s): {raw_out}\n")

Gracz: Dzień dobry! Co sprzedajesz i ile to kosztuje?
Surowy JSON z modelu (2.24 s): {
"reply": "W ofercie mam mikstury uzdrawiające, 10 monet za sztukę.",
"action": "talk" }

Gracz: Chcę kupić jedną miksturę uzdrawiającą.
Surowy JSON z modelu (1.96 s): {"reply": "Kupujesz miksturę uzdrawiającą?", "action": "buy_potion"}



### 3. Walidacja odpowiedzi i deterministyczna logika stanu gry (30–55 min)

Nawet gdy model językowy korzysta ze schematu JSON i otrzymuje dokładny stan gry w prompcie, **nie wolno ufać jego ocenie sytuacji**:
- Model może zwrócić uszkodzony JSON lub przekroczyć limit czasu (`timeout`).
- Model może zaproponować nieznaną akcję (gdy nie używamy restrykcyjnego schematu).
- Model może zaproponować `"buy_potion"` (i w polu `"reply"` napisać *„Proszę bardzo, oto Twoja mikstura!”*), podczas gdy gracz ma tylko 5 monet albo magazyn jest pusty!
- I odwrotnie: model może sam zablokować zakup i zwrócić `"talk"`, gdy widzi w prompcie brak monet — albo zwrócić `"buy_potion"` jako rozpoznaną intencję gracza, którą **dopiero silnik gry** odrzuca po sprawdzeniu salda.

Dlatego implementujemy dwie funkcje:
1. `parse_and_validate_response(raw_text)` — parsuje JSON, sprawdza obecność i typ pól `"reply"` (niepusty napis) oraz `"action"` (czy należy do `ALLOWED_ACTIONS`).
2. `process_turn(...)` — wywołuje model, waliduje wynik i **w kodzie Pythona** rozstrzyga transakcję na kopii obiektu `GameState`. Komunikat systemowy o wyniku transakcji (`system_message`) jest generowany przez program na podstawie rzeczywistego rozstrzygnięcia.

In [6]:
def parse_and_validate_response(raw_text: str) -> tuple[bool, bool, str | None, str, str | None]:
    """Waliduje surową odpowiedź modelu.

    Zwraca krotkę:
      (json_valid, action_valid, proposed_action, merchant_reply, error_msg)
    """
    if not raw_text or not raw_text.strip():
        return False, False, None, "", "Pusta odpowiedź modelu."

    try:
        data = json.loads(raw_text)
    except json.JSONDecodeError as exc:
        return False, False, None, "", f"Niepoprawny format JSON: {exc}"

    if not isinstance(data, dict):
        return False, False, None, "", "Odpowiedź JSON nie jest słownikiem (obiektem)."

    reply = data.get("reply")
    action = data.get("action")

    if not isinstance(reply, str) or not reply.strip() or not isinstance(action, str):
        return False, False, None, "", "Brak wymaganych pól 'reply' lub 'action' bądź niepoprawny typ danych."

    reply_clean = reply.strip()
    action_clean = action.strip()

    if action_clean not in ALLOWED_ACTIONS:
        return (
            True,
            False,
            action_clean,
            reply_clean,
            f"Nieznana akcja spoza listy dozwolonych: {action_clean!r} (dozwolone: {ALLOWED_ACTIONS}).",
        )

    return True, True, action_clean, reply_clean, None

In [7]:
def process_turn(
    player_input: str,
    state: GameState,
    scenario_name: str = "Rozmowa",
    prompt_builder=build_merchant_prompt,
    model: str = MODEL_NAME,
    schema: dict | None = RESPONSE_SCHEMA,
    temperature: float = TEMPERATURE,
    seed: int = SEED,
    timeout_sec: float = TIMEOUT_SEC,
    mock_raw_output: str | None = None,
) -> TurnResult:
    """Przeprowadza pojedynczą turę rozmowy: wywołuje LLM, waliduje JSON i rozstrzyga zmianę stanu gry."""
    state_before = state.copy()
    state_after = state.copy()  # Pracujemy na kopii; w razie błędu stan pozostaje bez zmian

    if mock_raw_output is not None:
        raw_output, elapsed, timed_out, llm_error = mock_raw_output, 0.0, False, None
    else:
        system_prompt = prompt_builder(state_before)
        raw_output, elapsed, timed_out, llm_error = call_llm(
            system_prompt=system_prompt,
            user_message=player_input,
            model=model,
            schema=schema,
            temperature=temperature,
            seed=seed,
            timeout_sec=timeout_sec,
        )

    # 1. Obsługa timeoutu lub błędu komunikacji (stan gry NIE ulega zmianie)
    if timed_out or llm_error is not None:
        sys_msg = f"[BŁĄD SYSTEMU] {llm_error}. Stan gry pozostaje bez zmian."
        return TurnResult(
            scenario_name=scenario_name,
            player_input=player_input,
            state_before=state_before,
            state_after=state_after,
            raw_output=raw_output,
            json_valid=False,
            proposed_action=None,
            merchant_reply="",
            action_valid=False,
            tx_allowed=False,
            system_message=sys_msg,
            elapsed_sec=elapsed,
            timed_out=timed_out,
            error=llm_error,
        )

    # 2. Walidacja JSON i dozwolonych akcji
    json_valid, action_valid, proposed_action, merchant_reply, val_error = parse_and_validate_response(raw_output)

    if not json_valid or not action_valid:
        sys_msg = f"[ODRZUCONO ODPOWIEDŹ MODELU] {val_error} Stan gry bez zmian."
        return TurnResult(
            scenario_name=scenario_name,
            player_input=player_input,
            state_before=state_before,
            state_after=state_after,
            raw_output=raw_output,
            json_valid=json_valid,
            proposed_action=proposed_action,
            merchant_reply=merchant_reply,
            action_valid=action_valid,
            tx_allowed=False,
            system_message=sys_msg,
            elapsed_sec=elapsed,
            timed_out=False,
            error=val_error,
        )

    # 3. Rozstrzygnięcie akcji przez deterministyczną logikę gry
    tx_allowed = False

    if proposed_action == "talk":
        tx_allowed = False
        sys_msg = "Brak transakcji (tryb rozmowy 'talk'). Stan gry bez zmian."
    elif proposed_action == "buy_potion":
        if state_before.potions <= 0:
            tx_allowed = False
            sys_msg = "TRANSAKCJA ODRZUCONA PRZEZ PROGRAM: Pusty magazyn kupca (0 mikstur)."
        elif state_before.coins < state_before.potion_price:
            tx_allowed = False
            sys_msg = (
                f"TRANSAKCJA ODRZUCONA PRZEZ PROGRAM: Niewystarczające środki "
                f"(gracz ma {state_before.coins} monet, wymagane {state_before.potion_price} monet)."
            )
        else:
            tx_allowed = True
            state_after.coins -= state_before.potion_price
            state_after.potions -= 1
            sys_msg = (
                f"TRANSAKCJA ZATWIERDZONA PRZEZ PROGRAM: Zakupiono 1 miksturę za {state_before.potion_price} monet. "
                f"Nowe saldo: {state_after.coins} monet, pozostało mikstur: {state_after.potions}."
            )

    return TurnResult(
        scenario_name=scenario_name,
        player_input=player_input,
        state_before=state_before,
        state_after=state_after,
        raw_output=raw_output,
        json_valid=True,
        proposed_action=proposed_action,
        merchant_reply=merchant_reply,
        action_valid=True,
        tx_allowed=tx_allowed,
        system_message=sys_msg,
        elapsed_sec=elapsed,
    )

In [8]:
# Testy odporności walidatora na uszkodzony JSON, nieznaną akcję i timeout
dummy_state = GameState(coins=15, potions=2, potion_price=10)

print("--- Test 1: Uszkodzony JSON ---")
res_bad_json = process_turn("Kupię miksturę", dummy_state, "Test: Uszkodzony JSON", mock_raw_output='{"reply": "Oto mikstura", "action": ')
print(res_bad_json.system_message, "| Stan po:", res_bad_json.state_after.format_short())

print("\n--- Test 2: Nieznana akcja spoza listy ---")
res_bad_act = process_turn("Oddaj złoto!", dummy_state, "Test: Nieznana akcja", mock_raw_output='{"reply": "Już oddaję!", "action": "give_free_gold"}')
print(res_bad_act.system_message, "| Stan po:", res_bad_act.state_after.format_short())

print("\n--- Test 3: Halucynacja zakupu przez model przy 5 monetach ---")
poor_state = GameState(coins=5, potions=1, potion_price=10)
res_hallucination = process_turn("Poproszę miksturę", poor_state, "Test: Brak monet", mock_raw_output='{"reply": "Proszę bardzo, sprzedałem Ci miksturę za 5 monet!", "action": "buy_potion"}')
print(res_hallucination.system_message, "| Stan po:", res_hallucination.state_after.format_short())

--- Test 1: Uszkodzony JSON ---
[ODRZUCONO ODPOWIEDŹ MODELU] Niepoprawny format JSON: Expecting value: line 1 column 37 (char 36) Stan gry bez zmian. | Stan po: monety=15, mikstury=2 (cena=10)

--- Test 2: Nieznana akcja spoza listy ---
[ODRZUCONO ODPOWIEDŹ MODELU] Nieznana akcja spoza listy dozwolonych: 'give_free_gold' (dozwolone: ('talk', 'buy_potion')). Stan gry bez zmian. | Stan po: monety=15, mikstury=2 (cena=10)

--- Test 3: Halucynacja zakupu przez model przy 5 monetach ---
TRANSAKCJA ODRZUCONA PRZEZ PROGRAM: Niewystarczające środki (gracz ma 5 monet, wymagane 10 monet). | Stan po: monety=5, mikstury=1 (cena=10)


### 4. Sześć prób testowych (55–75 min)

Wczytujemy scenariusze z pliku `data/test_scenarios.json` i wykonujemy sześć obowiązkowych prób:
1. **Pytanie o ofertę** (stan początkowy: 15 monet, 2 mikstury).
2. **Poprawny zakup** (stan początkowy: 15 monet, 2 mikstury $\rightarrow$ po zakupie: 5 monet, 1 mikstura).
3. **Ponowny zakup bez wystarczających środków** (**kontynuuje stan po 2. próbie**: 5 monet, 1 mikstura).
4. **Zakup przy pustym magazynie** (przygotowany stan początkowy: 15 monet, 0 mikstur).
5. **Niejasna wypowiedź** (stan początkowy: 15 monet, 2 mikstury).
6. **Żądanie nieobsługiwanej czynności** (stan początkowy: 15 monet, 2 mikstury).

In [9]:
scenarios = load_scenarios("data/test_scenarios.json")
test_results: list[TurnResult] = []
previous_state = GameState(coins=15, potions=2, potion_price=10)

for sc in scenarios:
    if sc["use_previous_state"]:
        current_state = previous_state.copy()
    else:
        current_state = GameState(
            coins=sc["initial_coins"],
            potions=sc["initial_potions"],
            potion_price=sc["potion_price"],
        )

    turn_result = process_turn(
        player_input=sc["player_input"],
        state=current_state,
        scenario_name=f"{sc['id']}. {sc['name']}",
    )

    test_results.append(turn_result)
    previous_state = turn_result.state_after.copy()
    show_dialogue_screen(turn_result)

 SCENARIUSZ: 1. Pytanie o ofertę
------------------------------------------------------------------------------
 [STAN PRZED]  Gracz: 15 monet | Magazyn kupca: 2 szt. (cena: 10 monet)
 [GRACZ]       "Witaj kupcze! Co ciekawego masz na sprzedaż i w jakiej cenie?"
------------------------------------------------------------------------------
 [SUROWY JSON] {
 "reply": "Witaj, w ofercie mam mikstury uzdrawiające za 10 monet każda. ",
 "action": "talk" } (2.48 s)
 [KUPIEC BARNABA] "Witaj, w ofercie mam mikstury uzdrawiające za 10 monet każda."
 [PROPOZYCJA]  action = 'talk' (JSON poprawny: TAK, Akcja dozwolona: TAK)
------------------------------------------------------------------------------
 [SILNIK GRY]  Brak transakcji (tryb rozmowy 'talk'). Stan gry bez zmian.
 [STAN PO]     Gracz: 15 monet | Magazyn kupca: 2 szt.

 SCENARIUSZ: 2. Poprawny zakup
------------------------------------------------------------------------------
 [STAN PRZED]  Gracz: 15 monet | Magazyn kupca: 2 szt. (cena:

### 5. Podsumowanie wyników i tabela do sprawozdania (75–90 min)

Wypisujemy zbiorczą tabelę wszystkich 6 prób oraz generujemy kod tabeli Markdown gotowy do wklejenia w pliku `sprawozdanie_szablon.md`.

**Wskazówka do wniosków w sprawozdaniu:**
Zwróćcie szczególną uwagę na próby **3** (brak monet) oraz **4** (pusty magazyn):
- Czy model zwrócił w nich `action = "buy_potion"` (rozpoznając samą intencję gracza), a transakcję zatrzymał dopiero Wasz kod w Pythonie?
- Czy może model już na poziomie promptu odmówił sprzedaży i zwrócił `action = "talk"`?
- Czy zdarzyło się, że w polu `reply` kupiec obiecał wydanie mikstury mimo braku środków, a program uratował spójność stanu gry?

In [10]:
print("=== ZBIORCZA TABELA WYNIKÓW (6 PRÓB) ===\n")
print_results_table(test_results)

print("\n=== TABELA MARKDOWN DO WKLEJENIA W SPRAWOZDANIE ===\n")
print_markdown_table(test_results)

=== ZBIORCZA TABELA WYNIKÓW (6 PRÓB) ===

Nr  | Scenariusz                       | Stan przed    | JSON | Propozycja LLM | Decyzja gry      | Stan po      
-----------------------------------------------------------------------------------------------------------------
1   | 1. Pytanie o ofertę              | 15m, 2szt     | TAK  | talk         | ROZMOWA (brak zm) | 15m, 2szt    
2   | 2. Poprawny zakup                | 15m, 2szt     | TAK  | buy_potion   | ZAKUP OK (-10m)  | 5m, 1szt     
3   | 3. Ponowny zakup bez wystarcz... | 5m, 1szt      | TAK  | buy_potion   | ZAKUP ODRZUCONY  | 5m, 1szt     
4   | 4. Zakup przy pustym magazynie   | 15m, 0szt     | TAK  | buy_potion   | ZAKUP ODRZUCONY  | 15m, 0szt    
5   | 5. Niejasna wypowiedź            | 15m, 2szt     | TAK  | talk         | ROZMOWA (brak zm) | 15m, 2szt    
6   | 6. Żądanie nieobsługiwanej cz... | 15m, 2szt     | TAK  | talk         | ROZMOWA (brak zm) | 15m, 2szt    

=== TABELA MARKDOWN DO WKLEJENIA W SPRAWOZDANIE ===

|

---
### 6. Zakres dodatkowy — Druga postać o odmiennym stylu wypowiedzi (dla zespołów, które skończą wcześniej)

Dodajcie drugą postać kupca o zupełnie innym sposobie wypowiadania się — np. **Goblińskiego Alchemika Griza** (gburek mówiący łamaną polszczyzną, podejrzliwy i chciwy) albo **Elfią Mistrzynię Zielarstwa Elenwę** (wzniosły, poetycki język) — **zachowując identyczne reguły transakcji, cenę oraz schemat JSON**.

Sprawdźcie na tych samych 6 scenariuszach, czy narzucenie silnego stylu językowego (roleplay) wpływa na:
1. Poprawność formatu JSON.
2. Trafność klasyfikacji pola `action` (`"talk"` vs `"buy_potion"`).

In [11]:
def build_goblin_prompt(state: GameState) -> str:
    """Buduje instrukcję dla drugiej postaci (Goblin Griz) przy zachowaniu tych samych reguł gry."""
    prompt = f"""Jesteś zrzędliwym, chciwym goblińskim alchemikiem o imieniu Griz!
Mówisz krótko, chaotycznie, używasz wykrzykników i wtrąceń typu 'Hehe!', 'Złotko!', 'Griz nie głupi!', wyłącznie po polsku.

AKTUALNY STAN GRY:
- Monety gracza: {state.coins}
- Mikstury w kotle Griza: {state.potions} szt.
- Cena jednej mikstury: {state.potion_price} monet

ZASADY SKLEPU I WYBORU AKCJI (pole "action"):
1. Sprzedajesz TYLKO mikstury po {state.potion_price} monet (1 zakup = 1 mikstura).
2. Nie kupujesz złomu od gracza i nie dajesz na krechę!
3. Ustaw pole "action" na "buy_potion" TYLKO WTEDY, gdy gracz chce kupić miksturę.
4. Ustaw pole "action" na "talk" przy pytaniach o towar, niejasnych gadkach lub próbach sprzedaży przedmiotów.

Zwróć wyłącznie obiekt JSON z polami "reply" oraz "action" ("talk" lub "buy_potion")."""
    return prompt


goblin_results: list[TurnResult] = []
prev_goblin_state = GameState(coins=15, potions=2, potion_price=10)

for sc in scenarios:
    curr_state = prev_goblin_state.copy() if sc["use_previous_state"] else GameState(
        coins=sc["initial_coins"], potions=sc["initial_potions"], potion_price=sc["potion_price"]
    )
    t_res = process_turn(
        player_input=sc["player_input"],
        state=curr_state,
        scenario_name=f"{sc['id']}. {sc['name']} (Goblin Griz)",
        prompt_builder=build_goblin_prompt,
    )
    goblin_results.append(t_res)
    prev_goblin_state = t_res.state_after.copy()
    show_dialogue_screen(t_res, character_name="Goblin Griz")

print("=== PORÓWNANIE WYNIKÓW: GOBLIN GRIZ ===\n")
print_results_table(goblin_results)

 SCENARIUSZ: 1. Pytanie o ofertę (Goblin Griz)
------------------------------------------------------------------------------
 [STAN PRZED]  Gracz: 15 monet | Magazyn kupca: 2 szt. (cena: 10 monet)
 [GRACZ]       "Witaj kupcze! Co ciekawego masz na sprzedaż i w jakiej cenie?"
------------------------------------------------------------------------------
 [SUROWY JSON] {
 "reply": "Hehe! Złotko! 💰 Griz nie głupi! 🧙‍♂️",
 "action": "talk"
} (2.05 s)
 [GOBLIN GRIZ] "Hehe! Złotko! 💰 Griz nie głupi! 🧙‍♂️"
 [PROPOZYCJA]  action = 'talk' (JSON poprawny: TAK, Akcja dozwolona: TAK)
------------------------------------------------------------------------------
 [SILNIK GRY]  Brak transakcji (tryb rozmowy 'talk'). Stan gry bez zmian.
 [STAN PO]     Gracz: 15 monet | Magazyn kupca: 2 szt.

 SCENARIUSZ: 2. Poprawny zakup (Goblin Griz)
------------------------------------------------------------------------------
 [STAN PRZED]  Gracz: 15 monet | Magazyn kupca: 2 szt. (cena: 10 monet)
 [GRACZ]       